## Imports and Environment

In [1]:
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.llms.openai import OpenAI
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.graph_stores.neo4j import Neo4jGraphStore, Neo4jPropertyGraphStore

import pandas as pd
from neo4j import GraphDatabase
from transformers import AutoTokenizer, AutoProcessor, AutoModelForImageTextToText
from transformers import pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer

import pandas as pd, re, ast, textwrap

from datasets import Dataset
import datacompy
from dataclasses import dataclass

import ragas
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas import evaluate, EvaluationDataset

import openai
from neo4j import GraphDatabase
from pdf2image import convert_from_path

from dotenv import load_dotenv, find_dotenv
import torch
import os
import re
import json
from tqdm import tqdm
import gc
import nest_asyncio
import asyncio
import openai
from tqdm.asyncio import tqdm as async_tqdm
import time


nest_asyncio.apply()

# Project root path for Azure Sandpit environment
project_root_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/graphrag-project" 

# # Change the current working directory to the project root
os.chdir(project_root_path)

dotenv_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/graphrag-project/.env"

# --- For Local Development environment ---
# dotenv_path = find_dotenv()

# Load the .env file from the path that was found.
load_dotenv(dotenv_path=dotenv_path)

project_root = os.path.dirname(dotenv_path)

relative_data_dir = os.getenv("GRAPH_DATASET_DIR")

data_directory = os.path.join(project_root, relative_data_dir)

nest_asyncio.apply()

hf_token = os.getenv("HUGGINGFACE_TOKEN")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
graphrag_api_key = os.getenv("GRAPHRAG_API_KEY")

# CRITICAL STEP: Set the environment variable that RAGAS and LangChain expect
if graphrag_api_key:
    os.environ["OPENAI_API_KEY"] = graphrag_api_key
else:
    # This will stop the execution if the key isn't found, preventing the error
    raise ValueError("API Key not found. Make sure GRAPHRAG_API_KEY is set in your .env file.")

# --- Configuration ---
# Path to your GraphRAG output directory
output_dir = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/graphrag-project/output"

# Neo4j Credentials from your environment
URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_DATABASE")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
AUTH = (NEO4J_USER, NEO4J_PASSWORD)

# Neo4j Credentials
URI = os.getenv("NEO4J_URI")
AUTH = (os.getenv("NEO4J_DATABASE"), os.getenv("NEO4J_PASSWORD"))

# Check neo4j connectivity
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    # Ensure the connection is valid
    driver.verify_connectivity()
    print("Connection successful!")


print(f"✅ Project root automatically determined as: {project_root}")
print(f"✅ .env file loaded from: {dotenv_path}")
print(f"📁 Data directory set to: {data_directory}")


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Please note that you are missing the optional dependency: fugue. If you need to use this functionality it must be installed.
Please note that you are missing the optional dependency: snowflake. If you need to use this functionality it must be installed.
Please note that you are missing the optional dependency: spark. If you need to use this functionality it must be installed.


Connection successful!
✅ Project root automatically determined as: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/graphrag-project
✅ .env file loaded from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/graphrag-project/.env
📁 Data directory set to: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/graphrag-project/input/generate_questions_copy.pdf


## Ingestion

In [2]:
# --- Load Data from Parquet Files ---
entities_df = pd.read_parquet(os.path.join(output_dir, "entities.parquet"))
relationships_df = pd.read_parquet(os.path.join(output_dir, "relationships.parquet"))

# --- Ingestion Script ---
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    with driver.session() as session:
        # Clear the database
        print("Clearing the database...")
        session.run("MATCH (n) DETACH DELETE n")

        # Create constraints for faster merging
        session.run("CREATE CONSTRAINT entity_id IF NOT EXISTS FOR (n:Entity) REQUIRE n.id IS UNIQUE")

        # Ingest Entities
        print(f"Ingesting {len(entities_df)} entities...")
        for _, row in entities_df.iterrows():
            session.run("""
                MERGE (e:Entity {id: $id})
                SET e.title = $title,
                    e.type = $type,
                    e.description = $description,
                    e.degree = $degree
            """, **row.to_dict())

        # Ingest Relationships
        print(f"Ingesting {len(relationships_df)} relationships...")
        for _, row in relationships_df.iterrows():
            session.run("""
                MATCH (source:Entity {title: $source})
                MATCH (target:Entity {title: $target})
                MERGE (source)-[r:RELATIONSHIP]->(target)
                SET r.description = $description,
                    r.weight = $weight
            """, **row.to_dict())

print("Graph ingestion from GraphRAG output is complete.")

Clearing the database...
Ingesting 124 entities...
Ingesting 97 relationships...
Graph ingestion from GraphRAG output is complete.


## Initialise Models

In [3]:
load_dotenv()

model_path = os.getenv("LLAMA_3.1_8B_INSTRUCT_DIR")

# Verify the path was loaded and is valid
if not model_path or not os.path.exists(model_path):
    raise ValueError("MODEL_PATH not found or invalid. Check your .env file and path.")
else:
    print(f"Model path loaded from .env: {model_path}")

# Initialize the tokenizer to get the token ID for our stop sequence
tokenizer = AutoTokenizer.from_pretrained(model_path)
# The semicolon is our desired stop character. Get its token ID.
semicolon_token_id = tokenizer.convert_tokens_to_ids(";")

# Now, initialize the LLM with the correct stop condition
llm = HuggingFaceLLM(
    model_name=model_path,
    tokenizer_name=model_path,
    device_map="auto",
    model_kwargs={"token": hf_token, "dtype": torch.bfloat16},
    # Use 'eos_token_id' which is the correct parameter for this purpose
    generate_kwargs={
        "temperature": 0.1,
        "do_sample": True,
        # This tells the model to stop generating as soon as it outputs a semicolon
        "eos_token_id": semicolon_token_id,
    }
)

print("HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.")

openai_llm = OpenAI(
    api_key=graphrag_api_key,
    model="gpt-4o",
    temperature=0.0,
    # Uncomment to set a timeout
    # timeout=120.0,
)

print("OpenAI LLM initialized successfully.")

# Initialise Nanonets OCR for document parsing
ocr_model_path = os.getenv("NANONETS_OCR_S_DIR")

if not model_path or not os.path.exists(ocr_model_path):
    raise ValueError("MODEL_PATH not found or invalid. Check your .env file and path.")
else:
    print(f"Model path loaded from .env: {ocr_model_path}")

try:
    ocr_processor = AutoProcessor.from_pretrained(ocr_model_path)
    ocr_model = AutoModelForImageTextToText.from_pretrained(
        ocr_model_path,
        device_map="auto",
        dtype=torch.bfloat16 # Corrected argument
    )
    print(f"✅ Nanonets OCR model ('{ocr_model_path}') loaded successfully.")
except Exception as e:
    print(f"❌ Failed to load Nanonets OCR model. Error: {e}")

# Initialize Re-ranker LLM
reranker = SentenceTransformerRerank(
    model="BAAI/bge-reranker-v2-m3",
    top_n=5  # Number of nodes to return after re-ranking
)

Model path loaded from .env: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/models/Llama-3.1-8B-Instruct


Loading checkpoint shards: 100%|██████████| 4/4 [01:30<00:00, 22.60s/it]


HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.
OpenAI LLM initialized successfully.
Model path loaded from .env: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/models/Nanonets-OCR-s


Loading checkpoint shards: 100%|██████████| 4/4 [01:13<00:00, 18.44s/it]


✅ Nanonets OCR model ('/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/models/Nanonets-OCR-s') loaded successfully.


## Query Engine

In [15]:
def extract_entities_with_llama(question_str: str, llm_model) -> list[str]:
    """
    Uses a local LLM to extract key entities from a question.

    Args:
        question_str: The user's question.
        llm_model: The initialized HuggingFaceLLM model (Llama 3.1).

    Returns:
        A list of extracted entity strings.
    """
    prompt = f"""
    Based on the following user question, identify and extract the key entities.
    An entity is a specific person, organization, project, or concept.
    Return the entities as a comma-separated list. Do not add any other text or explanation.

    Question: "{question_str}"

    Entities:
    """

    # Get the model's response
    response = llm_model.complete(prompt)
    
    # Clean the output to get a simple list of entities
    entities_text = response.text.strip()
    
    # Split by comma and clean up each entity
    extracted_entities = [entity.strip() for entity in entities_text.split(',') if entity.strip()]
    
    print(f"LLM extracted entities: {extracted_entities}")
    return extracted_entities


def query_graph_for_context(question_str: str, driver, llm_model) -> str:
    """
    Finds entities in a question using an LLM and retrieves their graph neighborhood.

    Args:
        question_str: The user's question.
        driver: The Neo4j database driver.
        llm_model: The language model for entity extraction.

    Returns:
        A string containing the retrieved graph context.
    """
    # Use Llama 3.1 to identify key entities from the question
    entities_in_question = extract_entities_with_llama(question_str, llm_model)

    if not entities_in_question:
        return "No key entities were identified in the question by the LLM."

    # This Cypher query finds the entities and their 1-hop and 2-hop neighbors
    cypher_query = """
    MATCH (n:Entity)
    WHERE n.title IN $entities
    CALL {
        WITH n
        MATCH (n)-[r1]-(neighbor1)
        RETURN "Entity: " + n.title + ", Neighbor: " + neighbor1.title + ", Relationship: " + r1.description AS context
        UNION
        WITH n
        MATCH (n)-[]-(neighbor1)-[r2]-(neighbor2)
        RETURN "Entity: " + n.title + ", 2-hop Neighbor: " + neighbor2.title + ", Path: " + r2.description AS context
    }
    RETURN context
    """

    with driver.session() as session:
        formatted_query_for_log = cypher_query.replace("$entities", str(entities_in_question))
        results = session.run(cypher_query, entities=entities_in_question)
        context_list = [record["context"] for record in results]

    retrieved_context = "\n".join(context_list)
    return retrieved_context, formatted_query_for_log


def generate_answer_with_context(question: str, context: str, llm_model) -> str:
    """
    Uses an LLM to generate an answer based on the retrieved context.
    """
    prompt = f"""
    Based on the following context, please answer the user's question.

    Context:
    {context}

    Question: {question}

    Answer:
    """
    response = llm_model.complete(prompt)
    return response.text


# --- Example Usage ---
question = "What initiatives demonstrate HTX's commitment to innovation?"

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    # Pass the local `llm` model for entity extraction
    retrieved_context = query_graph_for_context(question, driver, llm)
    
    # Use the more powerful OpenAI model for the final answer generation
    final_answer = generate_answer_with_context(question, retrieved_context, openai_llm)
    
    print("\n--- Final Answer ---")
    print(final_answer)

Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


LLM extracted entities: ['HTX', 'innovation', 'initiatives', 'commitment\n    HTX', 'innovation', 'commitment\n    HTX', 'initiatives', 'innovation', 'commitment\n    HTX', 'commitment', 'innovation', 'initiatives\n    HTX', 'innovation', 'initiatives', 'commitment\n    HTX', 'commitment', 'initiatives', 'innovation\n    HTX', 'initiatives', 'commitment', 'innovation\n    HTX', 'innovation', 'commitment\n    HTX', 'commitment', 'innovation\n    HTX', 'innovation\n    HTX', 'commitment\n    HTX\n    innovation\n    commitment\n    initiatives\n    HTX', 'initiatives\n    HTX', 'commitment\n    HTX', 'innovation\n    HTX', 'innovation', 'commitment\n    HTX', 'commitment', 'innovation\n    HTX', 'innovation', 'initiatives\n    HTX', 'commitment', 'initiatives\n    HTX', 'initiatives', 'innovation\n    HTX', 'initiatives', 'commitment\n    HTX', 'commitment', 'initiatives\n    HTX', 'initiatives\n    HTX', 'commitment\n    HTX', 'innovation\n    HTX\n    HTX', 'initiatives\n    HTX', 'com

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL (n, n) { ... }} {position: line: 4, column: 5, offset: 57} for query: '\n    MATCH (n:Entity)\n    WHERE n.title IN $entities\n    CALL {\n        WITH n\n        MATCH (n)-[r1]-(neighbor1)\n        RETURN "Entity: " + n.title + ", Neighbor: " + neighbor1.title + ", Relationship: " + r1.description AS context\n        UNION\n        WITH n\n        MATCH (n)-[]-(neighbor1)-[r2]-(neighbor2)\n        RETURN "Entity: " + n.title + ", 2-hop Neighbor: " + neighbor2.title + ", Path: " + r2.description AS context\n    }\n    RETURN context\n    '



--- Final Answer ---
HTX's commitment to innovation is demonstrated through several initiatives:

1. **Hatch Innovation Centre**: HTX launched Hatch as its innovation centre, focusing on public safety and security technologies. This centre fosters innovation within the agency by nurturing and advancing technological solutions to enhance public safety and security measures.

2. **Participation in CES 2024**: HTX, along with its innovation centre Hatch, attended CES 2024 to engage with technology and innovation partners, showcasing its commitment to fostering relationships within the tech industry and exploring new advancements.

3. **HacX Hackathon**: HTX organized the HacX hackathon event in partnership with Microsoft, highlighting its role in fostering innovation and collaboration in technology development.

4. **5G Projects**: HTX is involved in multiple 5G projects, including collaborations with StarHub, IBM, and IMDA, aimed at enhancing the operational readiness of the Singapore C

## Benchmarking

In [11]:
@dataclass
class CypherMatch:
    """
    A standalone class to evaluate if two Cypher queries produce the exact same dataset.
    This version intelligently normalizes DataFrames and uses the correct datacompy API.
    """
    name: str = "cypher_match_datacompy"
    
    def __post_init__(self):
        self.uri = os.getenv("NEO4J_URI")
        self.auth = (os.getenv("NEO4J_DATABASE"), os.getenv("NEO4J_PASSWORD"))

    def _execute_query(self, query: str) -> pd.DataFrame:
        """Executes a query and returns a pandas DataFrame."""
        if not query or query == "Cypher query not found" or pd.isna(query):
            return pd.DataFrame()
        try:
            with GraphDatabase.driver(self.uri, auth=self.auth) as driver:
                records, _, keys = driver.execute_query(query)
                return pd.DataFrame(records, columns=keys)
        except Exception as e:
            print(f"[ERROR] CypherMatch failed to execute query. Error: {e}")
            return pd.DataFrame()

    def score(self, sample_dict: dict) -> float:
        """
        Calculates the score by performing a robust, normalized comparison of the
        DataFrames returned by the two queries.
        """
        generated_query = sample_dict.get("generated_cypher")
        ground_truth_query = sample_dict.get("ground_truth_cypher")

        print("\n--- Evaluating Sample for Data Equivalence ---")
        print(f"[DEBUG] Generated Query: {generated_query}")
        print(f"[DEBUG] Ground Truth Query: {ground_truth_query}")

        df_generated = self._execute_query(generated_query)
        df_ground_truth = self._execute_query(ground_truth_query)

        # 1. Handle Empty or Mismatched Shape Results
        if df_generated.empty:
            print("[DEBUG] Generated query correctly returned no results. Score: 0.0")
            return 0.0
        if df_generated.shape != df_ground_truth.shape:
            print(f"[DEBUG] Mismatch: DataFrame shapes are different. "
                  f"GT: {df_ground_truth.shape}, Gen: {df_generated.shape}. Score: 0.0")
            return 0.0

        # 2. Normalize DataFrames for Robust Comparison
        try:
            # Create copies to avoid modifying original dataframes
            df_gen_norm = df_generated.copy()
            df_gt_norm = df_ground_truth.copy()

            # Force column names to be identical for comparison
            df_gen_norm.columns = df_gt_norm.columns

            # Use all columns as the index for a full-dataset comparison
            join_cols = df_gt_norm.columns.tolist()

            print("[DEBUG] DataFrames normalized successfully for comparison.")

        except Exception as e:
            print(f"[ERROR] DataFrame normalization failed. Error: {e}")
            return 0.0

        # 3. Perform Final Comparison
        try:
            compare = datacompy.Compare(
                df_gt_norm,
                df_gen_norm,
                join_columns=join_cols,
                df1_name='ground_truth',
                df2_name='generated',
                ignore_spaces=True,
                ignore_case=True
            )

            
            is_match = compare.matches(ignore_extra_columns=False)
            score = 1.0 if is_match else 0.0
            
            if not is_match:
                print(f"[DEBUG] Datacompy Report (Mismatch Found):\n{compare.report()}")

            print(f"[DEBUG] Comparison complete. Match: {is_match}. Score: {score}")
            return score
        except Exception as e:
            print(f"[ERROR] Datacompy comparison failed unexpectedly. Error: {e}")
            return 0.0

In [18]:
# --- Config paths ---
load_dotenv(override=True) 
relative_benchmark_path = os.getenv("GRAPH_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("GRAPH_BENCHMARK_DATASET_DIR not set in .env")
BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(3)GraphRAG_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(output_dir, OUTPUT_FILENAME)


# --- Load and Prepare Data ---
if not os.path.isfile(BENCHMARK_FILE_PATH):
    raise FileNotFoundError(f"Benchmark file not found at: {BENCHMARK_FILE_PATH}")

# Load the benchmark CSV, automatically handling potential encoding errors
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    print("UTF-8 decoding failed. Retrying with 'latin1' encoding.")
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding='latin1')

# --- CONFIG: Set the number of rows to test ---
# Uncomment the line below for a quick test on a subset of data
# benchmark_df = benchmark_df.head(3)
print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")


# --- Generate Predictions ---
ragas_data = {
    "question": [], "answer": [], "contexts": [], "ground_truth": [],
    "generated_cypher": [], "ground_truth_cypher": []
}

print("Starting data generation...")
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    for _, row in tqdm(benchmark_df.iterrows(), total=len(benchmark_df)):
        question = row["question"].strip()
        # Use .get() to safely access columns that might not exist in all rows
        gt_answer = row.get("gt_answer", "").strip()
        gt_cypher = row.get("gt_cypher", "").strip()

        # 1. Retrieve context and the generated Cypher query in a single call
        retrieved_contexts, gen_cypher = query_graph_for_context(question, driver, llm)
        
        # 2. Generate the final answer using the retrieved context
        final_answer = generate_answer_with_context(question, retrieved_contexts, openai_llm)
        
        # 3. Append all data for evaluation
        ragas_data["question"].append(question)
        ragas_data["answer"].append(final_answer)
        # Ensure contexts is a list of strings for Ragas compatibility
        ragas_data["contexts"].append([retrieved_contexts])
        ragas_data["ground_truth"].append(gt_answer)
        ragas_data["generated_cypher"].append(gen_cypher)
        ragas_data["ground_truth_cypher"].append(gt_cypher)

print("Data generation complete.")
unified_df = pd.DataFrame(ragas_data)

# --- Evaluation ---

# 1. Evaluate with Ragas (Answer Relevancy only)
print("\\n--- Starting Ragas Evaluation (answer_relevancy) ---")
ragas_dataset = Dataset.from_pandas(unified_df)
ragas_results = evaluate(
    ragas_dataset,
    metrics=[answer_relevancy],
)
ragas_df = ragas_results.to_pandas()
print("--- Ragas Evaluation Complete ---")

# 2. Evaluate with CypherMatch
print("\\n--- Starting Manual Evaluation (cypher_match) ---")
cypher_match_scores = []
cypher_match_metric = CypherMatch()

for sample in unified_df.to_dict('records'):
    score = cypher_match_metric.score(sample)
    cypher_match_scores.append(score)

print("--- Manual Evaluation Complete ---")

# 3. Combine all results into a final report
final_report_df = unified_df.copy()
# Add the scores from the two evaluations
final_report_df['answer_relevancy'] = ragas_df['answer_relevancy']
final_report_df['cypher_match'] = cypher_match_scores

# 4. Display and save the final, complete report
print("\\n--- Unified Evaluation Results ---")
display(final_report_df)

print("\\n--- Average Scores ---")
avg_scores = final_report_df[['answer_relevancy', 'cypher_match']].mean(numeric_only=True)
print(avg_scores)

final_report_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\\n✅ Successfully saved the final unified report to: {OUTPUT_FILE_PATH}")


UTF-8 decoding failed. Retrying with 'latin1' encoding.
Loaded 100 question-answer pairs for evaluation.
Starting data generation...


  0%|          | 0/100 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL (n, n) { ... }} {position: line: 4, column: 5, offset: 57} for query: '\n    MATCH (n:Entity)\n    WHERE n.title IN $entities\n    CALL {\n        WITH n\n        MATCH (n)-[r1]-(neighbor1)\n        RETURN "Entity: " + n.title + ", Neighbor: " + neighbor1.title + ", Relationship: " + r1.description AS context\n        UNION\n        WITH n\n        MATCH (n)-[]-(neighbor1)-[r2]-(neighbor2)\n        RETURN "Entity: " + n.title + ", 2-hop Neighbor: " + neighbor2.title + ", Path: " + r2.description AS context\n    }\n    RETURN context\n    '


LLM extracted entities: ['HTX', 'it\n    HTX is the only specific entity in the question. "it" is a pronoun and not a specific entity. Therefore', 'the correct list of entities is just HTX. \n\n    Answer:\n    HTX', 'Based on the following user question', 'identify and extract the key entities.\n    An entity is a specific person', 'organization', 'project', 'or concept.\n    Return the entities as a comma-separated list. Do not add any other text or explanation.\n\n    Question: "What is the purpose of the HTX project and who is the founder of the project?"\n\n    Entities:\n     HTX project', 'founder of the project\n    The entities in the question are the HTX project and the founder of the project. \n\n    Answer:\n    HTX project', 'founder of the project', 'Based on the following user question', 'identify and extract the key entities.\n    An entity is a specific person', 'organization', 'project', 'or concept.\n    Return the entities as a comma-separated list. Do not add any o

  1%|          | 1/100 [00:10<17:30, 10.61s/it]Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL (n, n) { ... }} {position: line: 4, column: 5, offset: 57} for query: '\n    MATCH (n:Entity)\n    WHERE n.title IN $entities\n    CALL {\n        WITH n\n        MATCH (n)-[r1]-(neighbor1)\n        RETURN "Entity: " + n.title + ", Neighbor: " + neighbor1.title + ", Relationship: " + r1.description AS context\n        UNION\n        WITH n\n        MATCH (n)-[]-(neighbor1)-[r2]-(neighbor2)\n        RETURN "Entity: " + n.title + ", 2-hop Neighbor: " + neighbor2.title + ", Path: " + r2.description AS context\n    }\n    RETURN context\n    '


LLM extracted entities: ['HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX\n    HTX']


...

[DEBUG] Mismatch: DataFrame shapes are different. GT: (0, 1), Gen: (99, 1). Score: 0.0

--- Evaluating Sample for Data Equivalence ---
[DEBUG] Generated Query: 
    MATCH (n:Entity)
    WHERE n.title IN ['HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX', 'CIO', 'Undaunted\n    HTX']
    CALL {
        WITH n
        MATCH (n)-[r1]-(neighbor1)
        RETURN "Entity: " + n.title + ", Neighbor: " + neighb

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: CALL subquery without a variable scope clause is now deprecated. Use CALL (n, n) { ... }} {position: line: 4, column: 5, offset: 853} for query: '\n    MATCH (n:Entity)\n    WHERE n.title IN [\'HTX\', \'Annual Walk & Run 2023\', \'flag-off\\n    HTX\', \'Annual Walk & Run 2023\', \'flag-off\\n    HTX\', \'Annual Walk & Run 2023\', \'flag-off\\n    HTX\', \'Annual Walk & Run 2023\', \'flag-off\\n    HTX\', \'Annual Walk & Run 2023\', \'flag-off\\n    HTX\', \'Annual Walk & Run 2023\', \'flag-off\\n    HTX\', \'Annual Walk & Run 2023\', \'flag-off\\n    HTX\', \'Annual Walk & Run 2023\', \'flag-off\\n    HTX\', \'Annual Walk & Run 2023\', \'flag-off\\n    HTX\', \'Annual Walk & Run 2023\', \'flag-off\\n    HTX\', \'Annual Walk & Run 2023\', \'flag-off

[DEBUG] Generated query correctly returned no results. Score: 0.0

--- Evaluating Sample for Data Equivalence ---
[DEBUG] Generated Query: 
    MATCH (n:Entity)
    WHERE n.title IN ['HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HTX', 'Annual Walk & Run 2023', 'flag-off\n    HT']
    CALL {
 

,question,answer,contexts,ground_truth,generated_cypher,ground_truth_cypher,answer_relevancy,cypher_match
0,What is HTX and who does it serve?,"HTX, also known as the Home Team Science and T...","[Entity: HTX, Neighbor: SINGAPORE POLICE FORCE...",HTX is Singapore's Home Team Science and Techn...,\n MATCH (n:Entity)\n WHERE n.title IN [...,MATCH (c:Concept)-[:HAS_DESCRIPTION]->(d:Descr...,0.930615,0.0
1,What is HTX's mission?,"I'm sorry, but the context provided does not i...",[],HTX's mission is to advance science and techno...,\n MATCH (n:Entity)\n WHERE n.title IN [...,MATCH (c:Concept)-[:HAS_DESCRIPTION]->(d:Descr...,0.000000,0.0
2,What is HTX's vision?,"I'm sorry, but the context you provided does n...",[],HTX's vision is to exponentially impact Singap...,\n MATCH (n:Entity)\n WHERE n.title IN [...,MATCH (c:Concept)-[:HAS_DESCRIPTION]->(d:Descr...,0.000000,0.0
3,Which capabilities does HTX focus on?,HTX focuses on capabilities related to homelan...,[],"HTX focuses on Digital (like C4I, cloud, cyber...",\n MATCH (n:Entity)\n WHERE n.title IN [...,MATCH (g:Grouping)-[:CONTAINS_CAPABILITY]->(c:...,0.903894,0.0
4,What does C4I mean in HTX's work?,The context provided does not explicitly defin...,"[Entity: HTX, Neighbor: SINGAPORE POLICE FORCE...","C4I stands for Command, Control, Communication...",\n MATCH (n:Entity)\n WHERE n.title IN [...,MATCH (c:Concept)-[:HAS_DESCRIPTION]->(d:Descr...,0.000000,0.0
...,...,...,...,...,...,...,...,...
95,Outline the route and format of HTX Annual Cyc...,The context provided does not include specific...,"[Entity: HTX, Neighbor: SINGAPORE POLICE FORCE...",The route ran from Road Safety Park to Nationa...,\n MATCH (n:Entity)\n WHERE n.title IN [...,MATCH (c:Concept)-[:HAS_DESCRIPTION]->(d:Descr...,0.000000,0.0
96,Which panel did HTX participate in with DHS S&...,HTX participated in a panel discussion with th...,"[Entity: HTX, Neighbor: SINGAPORE POLICE FORCE...",Hatch Centre Director Mok Shao Hong and CBP In...,\n MATCH (n:Entity)\n WHERE n.title IN [...,MATCH (c:Concept)-[:HAS_DESCRIPTION]->(d:Descr...,0.920947,0.0
97,What quote from HTX's CIO encapsulates the Und...,The context provided does not include a direct...,"[Entity: HTX, Neighbor: SINGAPORE POLICE FORCE...",Failure is part of success... No one succeeds ...,\n MATCH (n:Entity)\n WHERE n.title IN [...,MATCH (c:Concept)-[:HAS_DESCRIPTION]->(d:Descr...,0.000000,0.0
98,What dual outcomes does eXpresso! seek to achi...,"I'm sorry, but the context needed to answer th...",[],eXpresso! seeks to keep staff informed on HTX ...,\n MATCH (n:Entity)\n WHERE n.title IN [...,MATCH (c:Concept)-[:HAS_DESCRIPTION]->(d:Descr...,0.000000,0.0


\n--- Average Scores ---
answer_relevancy    0.56095
cypher_match        0.00000
dtype: float64
\n✅ Successfully saved the final unified report to: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/graphrag-project/output/(3)GraphRAG_benchmark_results.csv
